# Soft Prompt Fine-Tuning with Hugging Face PEFT
## Prompt Tuning • P-Tuning • Prefix Tuning

**Hands-on Jupyter lab — introductory to intermediate**

You will train three parameter-efficient adapters against the **same frozen pretrained causal language model**, on a small sentiment-label generation task. This lab follows the style of your earlier LoRA/QLoRA exercises: inspect the model, identify trainable tensors, train, test, save, reload, and compare.

> This is an educational experiment, **not** a production-quality sentiment classifier. With a tiny model/dataset, accuracy will vary and may be low. Use a stronger model and substantially more labeled data for a meaningful benchmark.

**Hardware:** CPU, NVIDIA CUDA, or Apple Silicon MPS (subject to compatible installed PyTorch/Transformers/PEFT versions). Unlike NF4 QLoRA, this exercise needs **no bitsandbytes quantization**. Internet is needed on the first run to download model/tokenizer.

## Learning objectives
1. Explain the difference between **hard tokens** and **soft/virtual tokens**.
2. Explain where Prompt Tuning, P-Tuning, and Prefix Tuning inject trainable information.
3. Use `PromptTuningConfig`, `PromptEncoderConfig`, and `PrefixTuningConfig`.
4. Freeze the pretrained base model while training PEFT prompt parameters.
5. Examine gradients, shapes, parameter counts, model outputs, and adapter checkpoints.

**Core difference from LoRA:** LoRA updates low-rank matrices attached to existing linear projections; soft-prompt methods learn task-conditioning vectors or networks instead of `A @ B` weight corrections.

## 0. Runtime setup

For **Google Colab**, first select **Runtime → Change runtime type → GPU** if available. A small CPU test is possible. Restart the kernel if you just upgraded core libraries.

Package versions change: if an error is due to a version mismatch, consult [PEFT prompt tuning](https://huggingface.co/docs/peft/main/package_reference/prompt_tuning), [P-tuning](https://huggingface.co/docs/peft/main/package_reference/p_tuning), and [prefix tuning](https://huggingface.co/docs/peft/main/package_reference/prefix_tuning).

In [ ]:
%pip install -q -U "torch" "transformers" "peft" "accelerate" "safetensors"

In [ ]:
import os, random, math, copy, sys
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import (
    PromptTuningConfig, PromptTuningInit,
    PromptEncoderConfig, PrefixTuningConfig,
    TaskType, PeftModel, get_peft_model,
)
import transformers, peft

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else
    "cpu"
)
print("Python:", sys.version.split()[0]); print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__); print("PEFT:", peft.__version__)
print("Device:", DEVICE)

## 1. Mathematical intuition

Let input token IDs be `[t₁, t₂, ..., tₙ]` and their embeddings be `E(tᵢ)`.

**Prompt tuning:** learn a matrix `P ∈ ℝ^(v × d)` (v virtual tokens, d embedding width), and feed `[P, E(t₁), ..., E(tₙ)]` to the **frozen** transformer. Each entry of P is a trainable scalar, like `nn.Parameter` in your LoRA lab.

**P-tuning:** learn an **encoder** (usually MLP, configurable) that produces input prompt embeddings. Unlike basic prompt tuning, learned prompts come from a reparameterization network.

**Prefix tuning:** learn virtual **key/value attention states** (or a network generating those states) for the transformer layers. It is not simply another string concatenated onto the text. For a causal LM with `L` layers, `H` attention heads, head size `D`, prefix length `v`, the per-batch key/value prefix conceptually has shape `[L, 2, batch, H, v, D]` (exact storage implementation varies).

In **all three**, the pretrained language-model weights stay frozen; *which new prompt parameters* are trainable differs.

| Method | Learned object | Where it acts | PEFT configuration |
|---|---|---|---|
| Prompt Tuning | Virtual token embedding table | Input embeddings | `PromptTuningConfig` |
| P-Tuning | Prompt embeddings via MLP/LSTM prompt encoder | Input embeddings | `PromptEncoderConfig` |
| Prefix Tuning | Key/value prefixes (optionally with projection) | Attention layers | `PrefixTuningConfig` |

**Terminology:** Hugging Face calls the prompt-encoder method *P-Tuning*. **P-Tuning v2** is related to deep prompt approaches, but is **not automatically the same API or architecture** as `PromptEncoderConfig`.

## 2. Choose one shared pretrained model

For quick smoke tests use `sshleifer/tiny-gpt2` (very small and **not suitable for strong accuracy**). For a more meaningful comparison use `distilgpt2` (larger and slower), especially on GPU. GPT-2-type models are selected because all three PEFT methods support a causal language modeling workflow, including prefix key/value caching.

`MODEL_ID` is deliberately a single variable, so the model is identical across experiments.

In [ ]:
MODEL_ID = "sshleifer/tiny-gpt2"  # fast demonstration
# MODEL_ID = "distilgpt2"        # better language-model capacity; slower
NUM_VIRTUAL_TOKENS = 8
MAX_REVIEW_TOKENS = 72
BATCH_SIZE = 4
EPOCHS = 2
LEARNING_RATE = 3e-3

# GPT-2 has no dedicated pad token. Reuse EOS token for padding.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def load_base():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    base.config.pad_token_id = tokenizer.pad_token_id
    base.config.use_cache = False  # recommended during training
    return base

base = load_base()
print("Hidden embedding size:", base.config.n_embd)
print("Transformer blocks:", base.config.n_layer)
print("Heads:", base.config.n_head)
print("Total base parameters:", sum(p.numel() for p in base.parameters()))
del base

## 3. Create a small labeled dataset

We teach the model to generate the literal class label `positive` or `negative` after a review. **The target is the answer tokens only**, not the entire review. This is crucial: otherwise training mostly teaches the model to reproduce the review text.

For reproducibility, the training and validation examples are explicitly included; no external dataset is required. To expand the experiment, replace them with the Hugging Face IMDB dataset, keeping the same preprocessing function.

In [ ]:
train_examples = [
    ("A delightful movie with excellent acting and a moving story.", "positive"),
    ("Wonderful performances and a beautiful soundtrack.", "positive"),
    ("I enjoyed every minute of this charming film.", "positive"),
    ("An inspiring and thoughtful movie worth watching.", "positive"),
    ("Brilliant direction and convincing characters.", "positive"),
    ("The film is funny, warm, and genuinely entertaining.", "positive"),
    ("A fantastic story with a satisfying ending.", "positive"),
    ("Great visuals and a memorable performance.", "positive"),
    ("The plot is dull and the acting is terrible.", "negative"),
    ("A disappointing film with very poor dialogue.", "negative"),
    ("I regret spending time watching this boring movie.", "negative"),
    ("The characters are annoying and the story makes no sense.", "negative"),
    ("Awful pacing and an extremely weak ending.", "negative"),
    ("A badly written and painfully slow film.", "negative"),
    ("Unconvincing acting and a confusing plot.", "negative"),
    ("This was tedious, predictable, and not enjoyable.", "negative"),
]
validation_examples = [
    ("An excellent, uplifting, and enjoyable film.", "positive"),
    ("Very entertaining with strong performances.", "positive"),
    ("Boring story and disappointing acting.", "negative"),
    ("A terrible movie that I would not recommend.", "negative"),
]

def make_prompt(review):
    return f"Review: {review}\nSentiment:"

print(make_prompt(train_examples[0][0]), train_examples[0][1])

### 3.1 Build causal-LM labels correctly

For `input_ids = [prompt_token_ids, label_token_ids, EOS]`, corresponding training labels are `[-100, ..., -100, label_token_ids, EOS]`. PyTorch ignores `-100` positions when computing cross entropy. The causal LM automatically shifts logits/labels internally when given `labels=...`.

PEFT adds virtual tokens internally. **Do not manually prepend fake token IDs** or manually add virtual-token labels when using `get_peft_model` — PEFT handles those adjustments.

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, examples):
        self.records = []
        for review, sentiment in examples:
            prompt_ids = tokenizer(
                make_prompt(review), add_special_tokens=False,
                truncation=True, max_length=MAX_REVIEW_TOKENS
            )["input_ids"]
            # Leading space is intentional for GPT-2 tokenization.
            answer_ids = tokenizer(
                " " + sentiment, add_special_tokens=False
            )["input_ids"]
            ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
            labels = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]
            assert len(ids) == len(labels)
            self.records.append({"input_ids": ids, "labels": labels})

    def __len__(self): return len(self.records)
    def __getitem__(self, i): return self.records[i]


def pad_collate(batch):
    width = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        n = len(ex["input_ids"])
        pad = width - n
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * pad)
        attention_mask.append([1] * n + [0] * pad)
        labels.append(ex["labels"] + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_ds = SentimentDataset(train_examples)
val_ds = SentimentDataset(validation_examples)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate)

batch = next(iter(train_loader))
for key, value in batch.items(): print(key, value.shape)
print("First label positions:", batch["labels"][0].tolist()[-12:])

## 4. Build the three PEFT configurations

All three use `TaskType.CAUSAL_LM` and `NUM_VIRTUAL_TOKENS`, but train different parameters. We initialize each from a **fresh copy** of the same pretrained checkpoint so experiments do not contaminate one another.

- Prompt tuning: random virtual prompt vectors; `num_virtual_tokens × hidden_size` trainable scalar values.
- P-Tuning: `encoder_hidden_size` controls the prompt encoder's intermediate dimension; the exact number of learned parameters depends on the PEFT encoder implementation.
- Prefix tuning: `prefix_projection=False` directly learns prefix states. With `prefix_projection=True`, an additional projection network is trained and parameter counts change.

Unlike your original LoRA lab's `alpha`, none of the three requires LoRA A/B matrices or a LoRA scaling factor.

In [ ]:
def make_peft_config(method):
    common = dict(task_type=TaskType.CAUSAL_LM,
                  num_virtual_tokens=NUM_VIRTUAL_TOKENS)
    if method == "prompt_tuning":
        return PromptTuningConfig(
            **common,
            prompt_tuning_init=PromptTuningInit.RANDOM,
        )
    if method == "p_tuning":
        return PromptEncoderConfig(
            **common,
            encoder_hidden_size=128,
            encoder_num_layers=2,
        )
    if method == "prefix_tuning":
        return PrefixTuningConfig(
            **common,
            prefix_projection=False,
        )
    raise ValueError(f"Unknown method: {method}")

METHODS = ["prompt_tuning", "p_tuning", "prefix_tuning"]
for method in METHODS:
    print(method, type(make_peft_config(method)).__name__)

## 5. Create PEFT models and inspect trainable tensors

`get_peft_model` wraps the pretrained causal LM. It freezes base weights and exposes prompt/prefix parameters for the optimizer. Check this empirically using `.named_parameters()` and `.requires_grad`. In particular, names/shape conventions can differ between PEFT versions; **inspection is more reliable than hard-coding an internal attribute path**.

In [ ]:
def build_method(method):
    model = get_peft_model(load_base(), make_peft_config(method))
    model = model.to(DEVICE)
    return model


def parameter_report(model):
    trainable = [(name, tuple(p.shape), p.numel())
                 for name, p in model.named_parameters() if p.requires_grad]
    for name, shape, count in trainable:
        print(f"{name:75s} shape={str(shape):22s} numel={count:,}")
    total_trainable = sum(x[2] for x in trainable)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {total_trainable:,} / {total:,} ({100*total_trainable/total:.5f}%)")
    return total_trainable

# Inspect one method first; the rest are trained sequentially later.
example_model = build_method("prompt_tuning")
example_model.print_trainable_parameters()
parameter_report(example_model)
del example_model
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 6. One forward pass and backpropagation

Like LoRA's A and B parameters, prompt parameters are updated by the optimizer after `.backward()`. Unlike LoRA, their shapes may be `[virtual_tokens, hidden_size]` or encoded key/value prefixes.

After `loss.backward()` we check that at least one prompt parameter has a gradient; base parameters should be frozen.

In [ ]:
model = build_method("prompt_tuning")
model.train()
batch = {k: v.to(DEVICE) for k, v in next(iter(train_loader)).items()}
model.zero_grad(set_to_none=True)
outputs = model(**batch)
print("Logits shape:", tuple(outputs.logits.shape))
print("Loss:", float(outputs.loss.detach().cpu()))
outputs.loss.backward()

for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, "shape", tuple(p.shape),
              "gradient?", p.grad is not None,
              "mean |grad|", float(p.grad.abs().mean()) if p.grad is not None else None)

del model, outputs
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 7. Define a shared training loop

The **same optimizer, epochs, batch size, and dataset** are used for all methods. This is an instructional comparison, not a formal controlled benchmark (e.g. model initialization and optimal learning rates can still differ). Training never updates the pretrained language model's weights.

On a slow machine, reduce `EPOCHS=1` or use the tiny GPT-2 checkpoint. For a useful accuracy study, switch to `distilgpt2`, use a larger dataset, and tune each method separately.

In [ ]:
def average_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            losses.append(loss.detach().float().cpu().item())
    return sum(losses) / len(losses)


def train_method(model, epochs=EPOCHS, lr=LEARNING_RATE):
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=lr
    )
    history = []
    for epoch in range(epochs):
        model.train()
        losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            losses.append(loss.detach().float().cpu().item())
        train_loss = sum(losses)/len(losses)
        val_loss = average_loss(model, val_loader)
        history.append({"epoch": epoch+1, "train_loss": train_loss,
                        "val_loss": val_loss})
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    return history

## 8. Train and save each method

The loop below **builds a new pretrained model each time**, trains only its prompt parameters, then saves an adapter checkpoint. It releases memory between methods. Checkpoints are stored under `soft_prompt_outputs/<method>`.

If you want to train **only one** method first, set `RUN_METHODS = ["prompt_tuning"]`. Once that works, run all three.

In [ ]:
RUN_METHODS = METHODS  # or ["prompt_tuning"] for a first test
OUTPUT_ROOT = "./soft_prompt_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
results = {}

for method in RUN_METHODS:
    print("\n" + "="*65)
    print("TRAINING:", method)
    print("="*65)
    torch.manual_seed(SEED)  # same seed for comparable initialization
    model = build_method(method)
    n_trainable = parameter_report(model)
    history = train_method(model)
    save_path = os.path.join(OUTPUT_ROOT, method)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    results[method] = {
        "trainable_params": n_trainable,
        "history": history,
        "adapter_path": save_path,
    }
    print("Saved adapter to", save_path)
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

print("All completed:", list(results))

## 9. Compare parameter efficiency and training losses

There is **no guaranteed winner**. Prompt tuning usually has particularly few trainable parameters; P-tuning's prompt encoder can have more; prefix-tuning parameter count depends on model layer/head dimensions. Low loss on 16 examples does not imply high real-world accuracy.

In [ ]:
print(f"{'Method':<18} {'Trainable':>12} {'Train loss':>12} {'Val loss':>12}")
for method, r in results.items():
    last = r["history"][-1]
    print(f"{method:<18} {r['trainable_params']:>12,} "
          f"{last['train_loss']:>12.4f} {last['val_loss']:>12.4f}")

## 10. Reload a saved soft-prompt adapter

Reload the *same* base architecture (`MODEL_ID`) first, then use `PeftModel.from_pretrained`. The adapter checkpoint is **not a standalone copy** of the base model. Set `is_trainable=True` only when resuming training; leave it false for inference.

In [ ]:
SELECT_METHOD = RUN_METHODS[0]  # e.g. "prefix_tuning"
base = load_base()
adapter_path = results[SELECT_METHOD]["adapter_path"]
loaded_model = PeftModel.from_pretrained(base, adapter_path)
loaded_model = loaded_model.to(DEVICE)
loaded_model.eval()
print("Loaded", SELECT_METHOD, "from", adapter_path)

## 11. Inference: generate a sentiment label

For a quick look, generate a short continuation. Tiny GPT-2 may emit irrelevant text or fail to output one of the labels; that reflects the model/data limitations and should not be hidden by claiming perfect accuracy.

Important: **do not manually include the virtual tokens** in the visible prompt. PEFT inserts them in the model computation.

In [ ]:
review = "This movie was wonderful and beautifully acted."
prompt = make_prompt(review)
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = loaded_model.generate(
        **inputs, max_new_tokens=4, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
print("Input:", prompt)
print("Generated continuation:", repr(tokenizer.decode(
    output_ids[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)))

## 12. Optional: score the two labels instead of free-form generation

For fairer label evaluation, compute the **conditional log-likelihood of the target answer tokens** with the same masked-label approach used in training. This avoids relying on arbitrary generated words.

The tiny dataset below is intentionally small; do not interpret its accuracy as a robust benchmark.

In [ ]:
@torch.no_grad()
def score_candidate(model, review, candidate):
    prompt_ids = tokenizer(
        make_prompt(review), return_tensors="pt", add_special_tokens=False
    )["input_ids"][0].tolist()
    ans_ids = tokenizer(" " + candidate, add_special_tokens=False)["input_ids"]
    ids = prompt_ids + ans_ids + [tokenizer.eos_token_id]
    labels = [-100]*len(prompt_ids) + ans_ids + [tokenizer.eos_token_id]
    inp = torch.tensor([ids], device=DEVICE)
    lab = torch.tensor([labels], device=DEVICE)
    mask = torch.ones_like(inp)
    # PEFT adjusts virtual-token labels internally.
    loss = model(input_ids=inp, attention_mask=mask, labels=lab).loss
    return -float(loss.detach().cpu())


def classify(model, review):
    scores = {label: score_candidate(model, review, label)
              for label in ["positive", "negative"]}
    return max(scores, key=scores.get), scores

correct = 0
for review, target in validation_examples:
    prediction, scores = classify(loaded_model, review)
    correct += prediction == target
    print(f"target={target:8s} prediction={prediction:8s} scores={scores}")
print("Demo validation accuracy:", correct / len(validation_examples))

# For rigorous comparison, score each method's saved checkpoint in turn.

## 13. What does `num_virtual_tokens=8` mean?

It represents **eight extra learned prompt positions**. These are **not eight new English words**, and `tokenizer.decode()` cannot generally turn them into meaningful text.

For prompt tuning, with hidden size `d`, the input soft-prompt table contains roughly `8 × d` trainable scalars. P-tuning includes encoder parameters. Prefix tuning learns attention key/value conditioning across model layers. None of these alters the tokenizer vocabulary or the model's original word embeddings.

## 14. Compare with your LoRA and QLoRA labs

| Property | LoRA | QLoRA | Prompt tuning | P-tuning | Prefix tuning |
|---|---|---|---|---|---|
| Base weights frozen? | Yes | Yes | Yes | Yes | Yes |
| Base weights quantized? | Optional | Usually 4-bit NF4 | Not required | Not required | Not required |
| Trains low-rank matrices? | Yes | Yes | No | No | No |
| Trains input prompt embeddings? | No | No | Yes | Encoder outputs them | No (attention KV prefix) |
| Extra task-dependent attention KV states? | Not as prefix | Not as prefix | No | No | Yes |

**Caution:** Prompt tuning, P-tuning and prefix tuning are often called *soft prompt methods* collectively, but their architectures are not interchangeable. Prefix tuning has more involved model-architecture compatibility requirements than basic prompt tuning.

## 15. Exercises

1. Change `NUM_VIRTUAL_TOKENS` from 8 to 4, 16, and 32. Track trainable parameter counts and validation loss.
2. Try `PromptTuningInit.TEXT` with a relevant instruction and `tokenizer_name_or_path=MODEL_ID`. Compare against RANDOM initialization.
3. Change `encoder_hidden_size` from 128 to 64 for P-Tuning; inspect changes in `numel()`.
4. Try `prefix_projection=True` and inspect which tensors become trainable.
5. Use `distilgpt2` instead of tiny GPT-2; note memory and speed.
6. Replace the examples with a real subset of `stanfordnlp/imdb`; reserve an untouched test split and compute class-label accuracy.
7. Compare checkpoint sizes across the three saved adapter directories.
8. After `.backward()`, verify the base model has no gradients while prompt parameters do.

**Practical warning:** Some PEFT/Transformers version combinations may have differences in `past_key_values` handling for prefix tuning. If prefix training or generation fails, first check the [official prefix-tuning docs](https://huggingface.co/docs/peft/main/package_reference/prefix_tuning) and use a compatible released version pair. The notebook is API-oriented and has not been executed against downloaded model weights in this environment.

## References (official docs)

- [Hugging Face PEFT — Prompt tuning](https://huggingface.co/docs/peft/main/package_reference/prompt_tuning)
- [Hugging Face PEFT — P-tuning](https://huggingface.co/docs/peft/main/package_reference/p_tuning)
- [Hugging Face PEFT — Prefix tuning](https://huggingface.co/docs/peft/main/package_reference/prefix_tuning)
- [Hugging Face cookbook — Prompt tuning with PEFT](https://huggingface.co/learn/cookbook/prompt_tuning_peft)

**Environment note:** Training requires download of the selected pretrained model. This notebook has been syntax-checked but not end-to-end executed with Hugging Face packages in the authoring environment.